In [1]:
2+2

4

In [3]:
import json
import os
import csv
import requests  # Add this import
import random
from datetime import datetime,timezone
from pathlib import Path
import deepseek_key
import hashlib
import time

# Get the API key
deep_key = deepseek_key.config_data["key"]

# ---------- Config ----------
STIMULI_PATH = Path("stimuli/stimuli.json")
PROMPTS_PATH = Path("prompts/prompts.json")
FEWSHOT_PATH = Path("prompts/few_shot_pool.json")

OUTPUT_CSV = Path("responses/responses_raw.csv")

MODEL_NAME = "deepseek-chat"
PROMPT_IDS_TO_RUN = ["direct_zero_v1", "cot_zero_v1", "direct_few_v1", "cot_few_v1"]

FEW_SHOT_POLICY_ID = "same_word_cross_sense"

N_RUNS_PER_CONDITION = 1   # If deterministic, no need for 3
TEMPERATURE = 0
MAX_TOKENS = 300

# DeepSeek API configuration
DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"

# ---------- Helpers ----------
def load_stimuli(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("stimuli.json must be a JSON array (a list of stimulus objects).")
    return data

def load_prompt_templates(path: Path) -> dict[str, dict]:
    """
    Returns mapping: prompt_id -> full prompt object
    """
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    prompts = data.get("prompts")
    if not isinstance(prompts, list):
        raise ValueError("prompts.json must contain a top-level key 'prompts'.")

    templates = {}
    for p in prompts:
        pid = p.get("prompt_id")
        if isinstance(pid, str):
            templates[pid] = p

    return templates

def load_fewshot_pool(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)
    

def build_few_shot_block(stimulus: dict, pool: dict, policy_id: str) -> str:
    policy = pool["policies"][policy_id]
    examples = pool["examples"]

    word = stimulus["target_word"]
    sense_id = stimulus["sense_id"]
    sentence = stimulus["sentence"]

    # Exclusion rule
    if policy.get("exclusions", {}).get("exclude_eval_sentence", False):
        examples = [e for e in examples if e["sentence"] != sentence]

    # Word filtering
    if policy["word_relation"] == "same_word":
        candidates = [e for e in examples if e["word"] == word]
    elif policy["word_relation"] == "cross_word":
        candidates = [e for e in examples if e["word"] != word]
    else:
        raise ValueError("Unknown word_relation")

    # Determine number of senses
    unique_senses = sorted(set(e["sense_id"] for e in candidates))
    num_senses = len(unique_senses)

    num_examples = policy["num_examples_by_num_senses"].get(str(num_senses))
    if not num_examples:
        raise ValueError(f"No num_examples defined for {num_senses} senses")

    # Deterministic sampling
    base_seed = policy.get("seed", 0)
    seed = deterministic_seed(stimulus, base_seed)
    rng = random.Random(seed)

    selected = []

    if policy["sense_balance"] in ["cross_sense_balanced", "balanced"]:
        per_sense = num_examples // num_senses

        for s in unique_senses:
            sense_examples = [e for e in candidates if e["sense_id"] == s]
            selected.extend(rng.sample(sense_examples, per_sense))

    else:
        selected = rng.sample(candidates, num_examples)

    # Build block
    block_parts = []
    for ex in selected:
        block_parts.append(
            f"Sentence: \"{ex['sentence']}\"\n"
            f"Question:\n"
            f"In this sentence, what does the word \"{ex['target_word']}\" mean?\n\n"
            f"Answer: {ex['meaning']}\n"
        )

    return "\n".join(block_parts)



def ensure_csv_has_header(path: Path, fieldnames: list[str]) -> None:
    if path.exists():
        return
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

def append_row(path: Path, fieldnames: list[str], row: dict) -> None:
    with path.open("a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)

def deterministic_seed(stimulus: dict, base_seed: int = 0) -> int:
    """
    Create a stable integer seed based on stimulus identity + base seed.
    """
    unique_string = f"{stimulus['prompt_id']}_{base_seed}"
    digest = hashlib.sha256(unique_string.encode("utf-8")).hexdigest()
    
    # Convert first 8 hex chars to int
    return int(digest[:8], 16)

from typing import Optional

def parse_structured_response(text: str) -> Optional[dict]:
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return None

    
def call_deepseek_api(prompt_text: str, retries: int = 3) -> str:
    headers = {
        "Authorization": f"Bearer {deep_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt_text}
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "stream": False
    }

    for attempt in range(retries):
        try:
            response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data, timeout=60)
            response.raise_for_status()
            result = response.json()
            return result["choices"][0]["message"]["content"]

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2)  # small pause before retry

    print("All retries failed.")
    return ""

# ---------- Main ----------
def main():
    # ---- Load inputs ----
    stimuli = load_stimuli(STIMULI_PATH)
    templates = load_prompt_templates(PROMPTS_PATH)
    pool = load_fewshot_pool(FEWSHOT_PATH)

    # ---- Verify prompt IDs exist ----
    for pid in PROMPT_IDS_TO_RUN:
        if pid not in templates:
            raise ValueError(f"prompt_id '{pid}' not found in prompts.json. Found: {list(templates.keys())}")

    # ---- CSV schema ----
    fieldnames = [
        "response_id",
        "timestamp_utc",
        "model",
        "temperature",
        "max_output_tokens",
        "prompt_id",
        "run_id",
        "target_word",
        "sense_id",
        "gold_sense",
        "sense_family",
        "context_type",
        "few_shot_policy_id",
        "query_mode",
        "sentence",
        "prompt_text",
        "response_text",
        "predicted_sense",
        "correct"
    ]
    ensure_csv_has_header(OUTPUT_CSV, fieldnames)

    # ---- Resume point ----

    total = len(stimuli) * len(PROMPT_IDS_TO_RUN) * N_RUNS_PER_CONDITION


    # ---- Run ----
    i = 0
    for stimulus in stimuli:
        for prompt_id in PROMPT_IDS_TO_RUN:
            prompt_obj = templates[prompt_id]
            template = prompt_obj["template"]
            prompt_parameters = prompt_obj.get("prompt_parameters", [])

            format_kwargs = dict(stimulus)

            # Inject few-shot block only when required
            if "few_shot_block" in prompt_parameters:
                few_shot_block = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)
                format_kwargs["few_shot_block"] = few_shot_block

            prompt_text = template.format(**format_kwargs)

            for run_id in range(1, N_RUNS_PER_CONDITION + 1):
                i += 1



                print(f"[{i}/{total}] {MODEL_NAME} | {stimulus.get('prompt_id')} | {prompt_id} | run {run_id}")

                response_text = call_deepseek_api(prompt_text)

                parsed = parse_structured_response(response_text)
                if parsed:
                    predicted_sense = parsed.get("sense_id")
                    correct = (predicted_sense == stimulus.get("sense_id"))
                else:
                    predicted_sense = None
                    correct = False

                row = {
                    "response_id": f"run_{i}",
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                    "model": MODEL_NAME,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_TOKENS,
                    "prompt_id": prompt_id,
                    "run_id": run_id,
                    "target_word": stimulus.get("target_word"),
                    "sense_id": stimulus.get("sense_id"),
                    "gold_sense": stimulus.get("gold_sense"),
                    "sense_family": stimulus.get("sense_family"),
                    "context_type": stimulus.get("context_type"),
                    "few_shot_policy_id": FEW_SHOT_POLICY_ID if "few_shot_block" in prompt_parameters else None,
                    "query_mode": "api",
                    "sentence": stimulus.get("sentence"),
                    "prompt_text": prompt_text,
                    "response_text": response_text,
                    "predicted_sense": predicted_sense,
                    "correct": correct
                }

                append_row(OUTPUT_CSV, fieldnames, row)

                # Gentle throttle to reduce connection resets
                import time
                time.sleep(0.5)

    print(f"\nDone. Saved to: {OUTPUT_CSV.resolve()}")

pool = load_fewshot_pool(FEWSHOT_PATH)
stimulus = load_stimuli(STIMULI_PATH)[0]

block1 = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)
block2 = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)

print("Deterministic:", block1 == block2)

if __name__ == "__main__":
    main()


Deterministic: True
[1/220] deepseek-chat | bank_river_01 | direct_zero_v1 | run 1
[2/220] deepseek-chat | bank_river_01 | cot_zero_v1 | run 1
[3/220] deepseek-chat | bank_river_01 | direct_few_v1 | run 1
[4/220] deepseek-chat | bank_river_01 | cot_few_v1 | run 1
[5/220] deepseek-chat | bank_river_02 | direct_zero_v1 | run 1
[6/220] deepseek-chat | bank_river_02 | cot_zero_v1 | run 1
[7/220] deepseek-chat | bank_river_02 | direct_few_v1 | run 1
[8/220] deepseek-chat | bank_river_02 | cot_few_v1 | run 1
[9/220] deepseek-chat | bank_river_03 | direct_zero_v1 | run 1
[10/220] deepseek-chat | bank_river_03 | cot_zero_v1 | run 1
[11/220] deepseek-chat | bank_river_03 | direct_few_v1 | run 1
[12/220] deepseek-chat | bank_river_03 | cot_few_v1 | run 1
[13/220] deepseek-chat | bank_money_01 | direct_zero_v1 | run 1
[14/220] deepseek-chat | bank_money_01 | cot_zero_v1 | run 1
[15/220] deepseek-chat | bank_money_01 | direct_few_v1 | run 1
[16/220] deepseek-chat | bank_money_01 | cot_few_v1 | run

In [2]:
import json
import os
import csv
import requests  # Add this import
import random
from datetime import datetime,timezone
from pathlib import Path
import deepseek_key
import hashlib
import time

# Get the API key
deep_key = deepseek_key.config_data["key"]

# ---------- Config ----------
STIMULI_PATH = Path("stimuli/ambiguous_stimuli.json")
PROMPTS_PATH = Path("prompts/prompts.json")
FEWSHOT_PATH = Path("prompts/few_shot_pool.json")
OUTPUT_CSV = Path("responses/ambiguous_responses_raw.csv")


MODEL_NAME = "deepseek-chat"
PROMPT_IDS_TO_RUN = ["direct_zero_v1", "cot_zero_v1", "direct_few_v1", "cot_few_v1"]

FEW_SHOT_POLICY_ID = "same_word_cross_sense"

N_RUNS_PER_CONDITION = 1   # If deterministic, no need for 3
TEMPERATURE = 0
MAX_TOKENS = 300

# DeepSeek API configuration
DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"

# ---------- Helpers ----------
def load_stimuli(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("stimuli.json must be a JSON array (a list of stimulus objects).")
    return data

def load_prompt_templates(path: Path) -> dict[str, dict]:
    """
    Returns mapping: prompt_id -> full prompt object
    """
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    prompts = data.get("prompts")
    if not isinstance(prompts, list):
        raise ValueError("prompts.json must contain a top-level key 'prompts'.")

    templates = {}
    for p in prompts:
        pid = p.get("prompt_id")
        if isinstance(pid, str):
            templates[pid] = p

    return templates

def load_fewshot_pool(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)
    

def build_few_shot_block(stimulus: dict, pool: dict, policy_id: str) -> str:
    policy = pool["policies"][policy_id]
    examples = pool["examples"]

    word = stimulus["target_word"]
    sense_id = stimulus["sense_id"]
    sentence = stimulus["sentence"]

    # Exclusion rule
    if policy.get("exclusions", {}).get("exclude_eval_sentence", False):
        examples = [e for e in examples if e["sentence"] != sentence]

    # Word filtering
    if policy["word_relation"] == "same_word":
        candidates = [e for e in examples if e["word"] == word]
    elif policy["word_relation"] == "cross_word":
        candidates = [e for e in examples if e["word"] != word]
    else:
        raise ValueError("Unknown word_relation")

    # Determine number of senses
    unique_senses = sorted(set(e["sense_id"] for e in candidates))
    num_senses = len(unique_senses)

    num_examples = policy["num_examples_by_num_senses"].get(str(num_senses))
    if not num_examples:
        raise ValueError(f"No num_examples defined for {num_senses} senses")

    # Deterministic sampling
    base_seed = policy.get("seed", 0)
    seed = deterministic_seed(stimulus, base_seed)
    rng = random.Random(seed)

    selected = []

    if policy["sense_balance"] in ["cross_sense_balanced", "balanced"]:
        per_sense = num_examples // num_senses

        for s in unique_senses:
            sense_examples = [e for e in candidates if e["sense_id"] == s]
            selected.extend(rng.sample(sense_examples, per_sense))

    else:
        selected = rng.sample(candidates, num_examples)

    # Build block
    block_parts = []
    for ex in selected:
        block_parts.append(
            f"Sentence: \"{ex['sentence']}\"\n"
            f"Question:\n"
            f"In this sentence, what does the word \"{ex['target_word']}\" mean?\n\n"
            f"Answer: {ex['meaning']}\n"
        )

    return "\n".join(block_parts)



def ensure_csv_has_header(path: Path, fieldnames: list[str]) -> None:
    if path.exists():
        return
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

def append_row(path: Path, fieldnames: list[str], row: dict) -> None:
    with path.open("a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)

def deterministic_seed(stimulus: dict, base_seed: int = 0) -> int:
    """
    Create a stable integer seed based on stimulus identity + base seed.
    """
    unique_string = f"{stimulus['prompt_id']}_{base_seed}"
    digest = hashlib.sha256(unique_string.encode("utf-8")).hexdigest()
    
    # Convert first 8 hex chars to int
    return int(digest[:8], 16)

from typing import Optional

def parse_structured_response(text: str) -> Optional[dict]:
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return None

    
def call_deepseek_api(prompt_text: str, retries: int = 3) -> str:
    headers = {
        "Authorization": f"Bearer {deep_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt_text}
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "stream": False
    }

    for attempt in range(retries):
        try:
            response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data, timeout=60)
            response.raise_for_status()
            result = response.json()
            return result["choices"][0]["message"]["content"]

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2)  # small pause before retry

    print("All retries failed.")
    return ""

# ---------- Main ----------
def main():
    # ---- Load inputs ----
    stimuli = load_stimuli(STIMULI_PATH)
    templates = load_prompt_templates(PROMPTS_PATH)
    pool = load_fewshot_pool(FEWSHOT_PATH)

    # ---- Verify prompt IDs exist ----
    for pid in PROMPT_IDS_TO_RUN:
        if pid not in templates:
            raise ValueError(f"prompt_id '{pid}' not found in prompts.json. Found: {list(templates.keys())}")

    # ---- CSV schema ----
    fieldnames = [
        "response_id",
        "timestamp_utc",
        "model",
        "temperature",
        "max_output_tokens",
        "prompt_id",
        "run_id",
        "target_word",
        "sense_id",
        "gold_sense",
        "sense_family",
        "context_type",
        "few_shot_policy_id",
        "query_mode",
        "sentence",
        "prompt_text",
        "response_text",
        "predicted_sense",
        "correct"
    ]
    ensure_csv_has_header(OUTPUT_CSV, fieldnames)

    # ---- Resume point ----

    total = len(stimuli) * len(PROMPT_IDS_TO_RUN) * N_RUNS_PER_CONDITION


    # ---- Run ----
    i = 0
    for stimulus in stimuli:
        for prompt_id in PROMPT_IDS_TO_RUN:
            prompt_obj = templates[prompt_id]
            template = prompt_obj["template"]
            prompt_parameters = prompt_obj.get("prompt_parameters", [])

            format_kwargs = dict(stimulus)

            # Inject few-shot block only when required
            if "few_shot_block" in prompt_parameters:
                few_shot_block = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)
                format_kwargs["few_shot_block"] = few_shot_block

            prompt_text = template.format(**format_kwargs)

            for run_id in range(1, N_RUNS_PER_CONDITION + 1):
                i += 1



                print(f"[{i}/{total}] {MODEL_NAME} | {stimulus.get('prompt_id')} | {prompt_id} | run {run_id}")

                response_text = call_deepseek_api(prompt_text)

                parsed = parse_structured_response(response_text)
                if parsed:
                    predicted_sense = parsed.get("sense_id")
                    correct = (predicted_sense == stimulus.get("sense_id"))
                else:
                    predicted_sense = None
                    correct = False

                row = {
                    "response_id": f"run_{i}",
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                    "model": MODEL_NAME,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_TOKENS,
                    "prompt_id": prompt_id,
                    "run_id": run_id,
                    "target_word": stimulus.get("target_word"),
                    "sense_id": stimulus.get("sense_id"),
                    "gold_sense": stimulus.get("gold_sense"),
                    "sense_family": stimulus.get("sense_family"),
                    "context_type": stimulus.get("context_type"),
                    "few_shot_policy_id": FEW_SHOT_POLICY_ID if "few_shot_block" in prompt_parameters else None,
                    "query_mode": "api",
                    "sentence": stimulus.get("sentence"),
                    "prompt_text": prompt_text,
                    "response_text": response_text,
                    "predicted_sense": predicted_sense,
                    "correct": correct
                }

                append_row(OUTPUT_CSV, fieldnames, row)

                # Gentle throttle to reduce connection resets
                import time
                time.sleep(0.5)

    print(f"\nDone. Saved to: {OUTPUT_CSV.resolve()}")

pool = load_fewshot_pool(FEWSHOT_PATH)
stimulus = load_stimuli(STIMULI_PATH)[0]

block1 = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)
block2 = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)

print("Deterministic:", block1 == block2)

if __name__ == "__main__":
    main()


Deterministic: True
[1/52] deepseek-chat | bank_river_01 | direct_zero_v1 | run 1
[2/52] deepseek-chat | bank_river_01 | cot_zero_v1 | run 1
[3/52] deepseek-chat | bank_river_01 | direct_few_v1 | run 1
[4/52] deepseek-chat | bank_river_01 | cot_few_v1 | run 1
[5/52] deepseek-chat | bank_river_02 | direct_zero_v1 | run 1
[6/52] deepseek-chat | bank_river_02 | cot_zero_v1 | run 1
[7/52] deepseek-chat | bank_river_02 | direct_few_v1 | run 1
[8/52] deepseek-chat | bank_river_02 | cot_few_v1 | run 1
[9/52] deepseek-chat | pitch_throw_01 | direct_zero_v1 | run 1
[10/52] deepseek-chat | pitch_throw_01 | cot_zero_v1 | run 1
[11/52] deepseek-chat | pitch_throw_01 | direct_few_v1 | run 1
[12/52] deepseek-chat | pitch_throw_01 | cot_few_v1 | run 1
[13/52] deepseek-chat | pitch_communication_01 | direct_zero_v1 | run 1
[14/52] deepseek-chat | pitch_communication_01 | cot_zero_v1 | run 1
[15/52] deepseek-chat | pitch_communication_01 | direct_few_v1 | run 1
[16/52] deepseek-chat | pitch_communicati

In [ ]:
import json
import os
import csv
import requests  # Add this import
import random
from datetime import datetime,timezone
from pathlib import Path
import deepseek_key
import hashlib
import time

# Get the API key
deep_key = deepseek_key.config_data["key"]

# ---------- Config ----------
STIMULI_PATH = Path("stimuli/deepseek_rerun_ambiguous_stimuli.json")
PROMPTS_PATH = Path("prompts/prompts.json")
FEWSHOT_PATH = Path("prompts/few_shot_pool.json")

OUTPUT_CSV = Path("responses/long_ambiguous_responses_raw.csv")


MODEL_NAME = "deepseek-chat"
PROMPT_IDS_TO_RUN = ["direct_zero_v1", "cot_zero_v1", "direct_few_v1", "cot_few_v1"]

FEW_SHOT_POLICY_ID = "same_word_cross_sense"

N_RUNS_PER_CONDITION = 1   # If deterministic, no need for 3
TEMPERATURE = 0
MAX_TOKENS = 500

# DeepSeek API configuration
DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"

# ---------- Helpers ----------
def load_stimuli(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("stimuli.json must be a JSON array (a list of stimulus objects).")
    return data

def load_prompt_templates(path: Path) -> dict[str, dict]:
    """
    Returns mapping: prompt_id -> full prompt object
    """
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    prompts = data.get("prompts")
    if not isinstance(prompts, list):
        raise ValueError("prompts.json must contain a top-level key 'prompts'.")

    templates = {}
    for p in prompts:
        pid = p.get("prompt_id")
        if isinstance(pid, str):
            templates[pid] = p

    return templates

def load_fewshot_pool(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)
    

def build_few_shot_block(stimulus: dict, pool: dict, policy_id: str) -> str:
    policy = pool["policies"][policy_id]
    examples = pool["examples"]

    word = stimulus["target_word"]
    sense_id = stimulus["sense_id"]
    sentence = stimulus["sentence"]

    # Exclusion rule
    if policy.get("exclusions", {}).get("exclude_eval_sentence", False):
        examples = [e for e in examples if e["sentence"] != sentence]

    # Word filtering
    if policy["word_relation"] == "same_word":
        candidates = [e for e in examples if e["word"] == word]
    elif policy["word_relation"] == "cross_word":
        candidates = [e for e in examples if e["word"] != word]
    else:
        raise ValueError("Unknown word_relation")

    # Determine number of senses
    unique_senses = sorted(set(e["sense_id"] for e in candidates))
    num_senses = len(unique_senses)

    num_examples = policy["num_examples_by_num_senses"].get(str(num_senses))
    if not num_examples:
        raise ValueError(f"No num_examples defined for {num_senses} senses")

    # Deterministic sampling
    base_seed = policy.get("seed", 0)
    seed = deterministic_seed(stimulus, base_seed)
    rng = random.Random(seed)

    selected = []

    if policy["sense_balance"] in ["cross_sense_balanced", "balanced"]:
        per_sense = num_examples // num_senses

        for s in unique_senses:
            sense_examples = [e for e in candidates if e["sense_id"] == s]
            selected.extend(rng.sample(sense_examples, per_sense))

    else:
        selected = rng.sample(candidates, num_examples)

    # Build block
    block_parts = []
    for ex in selected:
        block_parts.append(
            f"Sentence: \"{ex['sentence']}\"\n"
            f"Question:\n"
            f"In this sentence, what does the word \"{ex['target_word']}\" mean?\n\n"
            f"Answer: {ex['meaning']}\n"
        )

    return "\n".join(block_parts)



def ensure_csv_has_header(path: Path, fieldnames: list[str]) -> None:
    if path.exists():
        return
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

def append_row(path: Path, fieldnames: list[str], row: dict) -> None:
    with path.open("a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)

def deterministic_seed(stimulus: dict, base_seed: int = 0) -> int:
    """
    Create a stable integer seed based on stimulus identity + base seed.
    """
    unique_string = f"{stimulus['prompt_id']}_{base_seed}"
    digest = hashlib.sha256(unique_string.encode("utf-8")).hexdigest()
    
    # Convert first 8 hex chars to int
    return int(digest[:8], 16)

from typing import Optional

def parse_structured_response(text: str) -> Optional[dict]:
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return None

    
def call_deepseek_api(prompt_text: str, retries: int = 3) -> str:
    headers = {
        "Authorization": f"Bearer {deep_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt_text}
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "stream": False
    }

    for attempt in range(retries):
        try:
            response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data, timeout=60)
            response.raise_for_status()
            result = response.json()
            return result["choices"][0]["message"]["content"]

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2)  # small pause before retry

    print("All retries failed.")
    return ""

# ---------- Main ----------
def main():
    # ---- Load inputs ----
    stimuli = load_stimuli(STIMULI_PATH)
    templates = load_prompt_templates(PROMPTS_PATH)
    pool = load_fewshot_pool(FEWSHOT_PATH)

    # ---- Verify prompt IDs exist ----
    for pid in PROMPT_IDS_TO_RUN:
        if pid not in templates:
            raise ValueError(f"prompt_id '{pid}' not found in prompts.json. Found: {list(templates.keys())}")

    # ---- CSV schema ----
    fieldnames = [
        "response_id",
        "timestamp_utc",
        "model",
        "temperature",
        "max_output_tokens",
        "prompt_id",
        "run_id",
        "target_word",
        "sense_id",
        "gold_sense",
        "sense_family",
        "context_type",
        "few_shot_policy_id",
        "query_mode",
        "sentence",
        "prompt_text",
        "response_text",
        "predicted_sense",
        "correct"
    ]
    ensure_csv_has_header(OUTPUT_CSV, fieldnames)

    # ---- Resume point ----

    total = len(stimuli) * len(PROMPT_IDS_TO_RUN) * N_RUNS_PER_CONDITION


    # ---- Run ----
    i = 0
    for stimulus in stimuli:
        for prompt_id in PROMPT_IDS_TO_RUN:
            prompt_obj = templates[prompt_id]
            template = prompt_obj["template"]
            prompt_parameters = prompt_obj.get("prompt_parameters", [])

            format_kwargs = dict(stimulus)

            # Inject few-shot block only when required
            if "few_shot_block" in prompt_parameters:
                few_shot_block = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)
                format_kwargs["few_shot_block"] = few_shot_block

            prompt_text = template.format(**format_kwargs)

            for run_id in range(1, N_RUNS_PER_CONDITION + 1):
                i += 1



                print(f"[{i}/{total}] {MODEL_NAME} | {stimulus.get('prompt_id')} | {prompt_id} | run {run_id}")

                response_text = call_deepseek_api(prompt_text)

                parsed = parse_structured_response(response_text)
                if parsed:
                    predicted_sense = parsed.get("sense_id")
                    correct = (predicted_sense == stimulus.get("sense_id"))
                else:
                    predicted_sense = None
                    correct = False

                row = {
                    "response_id": f"run_{i}",
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                    "model": MODEL_NAME,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_TOKENS,
                    "prompt_id": prompt_id,
                    "run_id": run_id,
                    "target_word": stimulus.get("target_word"),
                    "sense_id": stimulus.get("sense_id"),
                    "gold_sense": stimulus.get("gold_sense"),
                    "sense_family": stimulus.get("sense_family"),
                    "context_type": stimulus.get("context_type"),
                    "few_shot_policy_id": FEW_SHOT_POLICY_ID if "few_shot_block" in prompt_parameters else None,
                    "query_mode": "api",
                    "sentence": stimulus.get("sentence"),
                    "prompt_text": prompt_text,
                    "response_text": response_text,
                    "predicted_sense": predicted_sense,
                    "correct": correct
                }

                append_row(OUTPUT_CSV, fieldnames, row)

                # Gentle throttle to reduce connection resets
                import time
                time.sleep(0.5)

    print(f"\nDone. Saved to: {OUTPUT_CSV.resolve()}")

pool = load_fewshot_pool(FEWSHOT_PATH)
stimulus = load_stimuli(STIMULI_PATH)[0]

block1 = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)
block2 = build_few_shot_block(stimulus, pool, FEW_SHOT_POLICY_ID)

print("Deterministic:", block1 == block2)

if __name__ == "__main__":
    main()


Deterministic: True
[1/20] deepseek-chat | bank_river_01 | direct_zero_v1 | run 1
[2/20] deepseek-chat | bank_river_01 | cot_zero_v1 | run 1
[3/20] deepseek-chat | bank_river_01 | direct_few_v1 | run 1
[4/20] deepseek-chat | bank_river_01 | cot_few_v1 | run 1
[5/20] deepseek-chat | bank_river_02 | direct_zero_v1 | run 1
[6/20] deepseek-chat | bank_river_02 | cot_zero_v1 | run 1
[7/20] deepseek-chat | bank_river_02 | direct_few_v1 | run 1
[8/20] deepseek-chat | bank_river_02 | cot_few_v1 | run 1
[9/20] deepseek-chat | pitch_communication_02 | direct_zero_v1 | run 1
[10/20] deepseek-chat | pitch_communication_02 | cot_zero_v1 | run 1
[11/20] deepseek-chat | pitch_communication_02 | direct_few_v1 | run 1
[12/20] deepseek-chat | pitch_communication_02 | cot_few_v1 | run 1
[13/20] deepseek-chat | ring_jewelry_01 | direct_zero_v1 | run 1
[14/20] deepseek-chat | ring_jewelry_01 | cot_zero_v1 | run 1
[15/20] deepseek-chat | ring_jewelry_01 | direct_few_v1 | run 1
[16/20] deepseek-chat | ring_j

: 